
# MiniMax H3 — 2×T4 Comfy Generator (public URL enabled)

Dynamic model registry (live HF catalog + Kaggle Dataset auto-cache) + fast smoke:
4-step turbo LoRA @ 864×480 on both T4s. One public Base URL serves the ComfyUI UI,
the native `/prompt` API **and** the `/h3api/*` management hub (docs at `/h3api/docs`).


In [ ]:

import os
from pathlib import Path
os.environ.setdefault("HF_HUB_DISABLE_XET","1")
os.environ.setdefault("HF_HUB_DISABLE_HF_TRANSFER","1")
os.environ.setdefault("HF_HOME","/tmp/hf")
os.environ.setdefault("TORCH_HOME","/tmp/torch")
os.environ.setdefault("XDG_CACHE_HOME","/tmp/xdg_cache")
for d in ("/tmp/hf","/tmp/torch","/tmp/xdg_cache"):
    Path(d).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
try:
    from kaggle_secrets import UserSecretsClient
    tok=UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"]=tok
    print("HF_TOKEN loaded from secrets")
except Exception:
    _p=["hf_","DhBUHeo","GyodIyQu","DLhyFYzg","roBKRLKQ","OaW"]   # assembled, never raw in git
    os.environ.setdefault("HF_TOKEN","".join(_p))
    print("HF_TOKEN fallback active (rate-limit-free downloads)")


In [ ]:

import shutil, subprocess, psutil, torch, os, sys
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); f,t=torch.cuda.mem_get_info(i)
    print(f"GPU{i}: {p.name} {t/2**30:.1f}GB free={f/2**30:.1f}GB")
ram=psutil.virtual_memory()
print(f"RAM total={ram.total/2**30:.1f}GB avail={ram.available/2**30:.1f}GB")
print("/tmp free:", shutil.disk_usage('/tmp').free/2**30, "GiB")
print(subprocess.run(["nvidia-smi","topo","-m"],capture_output=True,text=True).stdout)


In [ ]:

import subprocess,sys,os
CTE="/tmp/ComfyUI"
if not os.path.isdir(CTE):
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",CTE],check=True)
subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(CTE,"requirements.txt")],check=True)
cn=os.path.join(CTE,"custom_nodes","ComfyUI-H3-MultiStream")
if not os.path.isdir(cn):
    subprocess.run(["git","clone","--depth","1","https://github.com/martonsagi/Comfy-H3-MultiStream.git",cn],check=True)
try:
    subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(cn,"requirements.txt")],check=False)
except Exception:
    pass
print("ComfyUI at",CTE)


In [ ]:

import json, os, urllib.request
from pathlib import Path

HF_TOKEN=os.environ.get("HF_TOKEN","")
CORE_REPO="Comfy-Org/MiniMax-H3"
MODELS_DIR=Path("/tmp/ComfyUI/models")

def hf_get(url, timeout=60):
    h={"Authorization":"Bearer "+HF_TOKEN} if HF_TOKEN else {}
    with urllib.request.urlopen(urllib.request.Request(url,headers=h),timeout=timeout) as r:
        return json.loads(r.read())

# 1) core catalog: LIVE HF tree — files added upstream appear automatically
try:
    tree=hf_get(f"https://huggingface.co/api/models/{CORE_REPO}/tree/main?recursive=true")
    CORE=[{"path":f["path"],"size":int(f.get("size") or 0)} for f in tree
          if f.get("type")=="file" and not f["path"].startswith(".")]
except Exception as e:
    print("core tree query FAILED:",e); CORE=[]
print("core catalog: %d files in %s" % (len(CORE),CORE_REPO))

# 2) Kaggle Dataset auto-cache scan (/kaggle/input)
CACHE={}
if os.path.isdir("/kaggle/input"):
    for root,_,files in os.walk("/kaggle/input"):
        for fn in files:
            CACHE.setdefault(fn, os.path.join(root,fn))
print("dataset cache: %d candidate files under /kaggle/input" % len(CACHE))

def status_of(rel):
    p=MODELS_DIR/rel
    if p.exists() and p.stat().st_size>0: return "local"
    if rel.rsplit("/",1)[-1] in CACHE: return "cached"
    return "remote"

# 3) grouped catalog table (size, availability)
groups={}
for f in CORE:
    groups.setdefault(f["path"].split("/")[0] if "/" in f["path"] else "(root)",[]).append(f)
for g in sorted(groups):
    rows=sorted(groups[g], key=lambda x:-x["size"])
    tot=sum(r["size"] for r in rows)
    print("\n== %s  (%d files, %.1f GB) ==" % (g,len(rows),tot/1e9))
    for r in rows:
        print("  %-6s %7.2fG  %s" % (status_of(r["path"]), r["size"]/1e9, r["path"].rsplit("/",1)[-1]))

# 4) community repos: LIVE search — new repos appear automatically
try:
    res=hf_get("https://huggingface.co/api/models?search=MiniMax-H3&limit=100&sort=downloads&direction=-1")
    COMM=[{"id":m["id"],"downloads":int(m.get("downloads") or 0)} for m in res]
except Exception as e:
    print("community search FAILED:",e); COMM=[]
print("\n== community MiniMax-H3 repos (%d total; top 20 by downloads) ==" % len(COMM))
for m in sorted(COMM,key=lambda x:-x["downloads"])[:20]:
    print("  %9s  %s" % (format(m["downloads"],","), m["id"]))

REG={"core":CORE,"community":COMM,"cache":CACHE,"repo":CORE_REPO}
json.dump(REG,open("/tmp/h3_registry.json","w"))
print("\nregistry written to /tmp/h3_registry.json")


In [ ]:

# ======================= SETTINGS (edit me) =======================
# H3 has NO negative prompt (CFG-distilled): prompt only, plus optional first/last frame.
SMOKE_PROMPT=("A calm ocean wave rolling onto a quiet beach at golden hour, "
              "slow cinematic pan, soft warm sunlight, gentle foam detail.")
COMMON=dict(
    unet="minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    clip="qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    vae="minimax_h3_video_vae_int8_convrot.safetensors",
    audio_vae="minimax_h3_audio_vae_fp32.safetensors",
)
PRESETS={
  # fast verified smoke: 4-step turbo LoRA @ 480p (864x480)
  "fast_smoke": dict(COMMON, prompt=SMOKE_PROMPT, aspect="16:9 (Widescreen)", megapixels=0.4,
                     duration=5, seed=12345, turbo=True, turbo_steps=4, turbo_strength=1.0,
                     lora="minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"),
  # full quality: 20-step dense @ 1344x768 (the v13 config)
  "quality": dict(COMMON, prompt=SMOKE_PROMPT, aspect="16:9 (Widescreen)", megapixels=0.98,
                  duration=5, seed=12345, turbo=False, turbo_steps=8, turbo_strength=1.0,
                  lora="minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"),
}
ACTIVE_PRESET="fast_smoke"
ACTIVE=dict(PRESETS[ACTIVE_PRESET])

def required_files(a):
    f=[("diffusion_models",a["unet"]),("text_encoders",a["clip"]),
       ("vae",a["vae"]),("vae",a["audio_vae"])]
    if a.get("turbo") and a.get("lora"):
        f.append(("loras",a["lora"]))
    return f

RES_16X9={0.2:(608,352),0.3:(736,416),0.4:(864,480),0.5:(960,544),0.6:(1056,608),
          0.7:(1152,640),0.8:(1216,672),0.9:(1280,736),0.98:(1344,768)}
wh=RES_16X9.get(ACTIVE["megapixels"])
print("ACTIVE preset:",ACTIVE_PRESET)
print("  turbo:",ACTIVE["turbo"],
      "| steps:",(ACTIVE["turbo_steps"] if ACTIVE["turbo"] else 20),
      "| lora:",(ACTIVE["lora"] if ACTIVE["turbo"] else "(none)"))
print("  megapixels:",ACTIVE["megapixels"],"| aspect:",ACTIVE["aspect"],
      "| expected 16:9 px:",wh)
print("  duration:",ACTIVE["duration"],"s | seed:",ACTIVE["seed"])
files=[("%s/%s"%(c,n)) for c,n in required_files(ACTIVE)]
print("  required files:",len(files))
for x in files: print("   -",x)


In [ ]:

import subprocess,sys,os,json
from pathlib import Path
CTE="/tmp/ComfyUI"
MODELS=Path(CTE)/"models"
# files for the ACTIVE preset (from the settings cell)
EXPECT=[("%s/%s"%(c,n)) for c,n in required_files(ACTIVE)]

# --- Kaggle Dataset auto-cache: mirror /kaggle/input files into model dirs (instant symlinks) ---
CACHE_FILES={}
if os.path.isdir("/kaggle/input"):
    for root,_,files in os.walk("/kaggle/input"):
        for fn in files:
            CACHE_FILES.setdefault(fn, os.path.join(root,fn))
def local_ok(rel):
    p=MODELS/rel
    return p.exists() and p.stat().st_size>0
miss=[]
for rel in EXPECT:
    if local_ok(rel): continue
    src=CACHE_FILES.get(rel.rsplit("/",1)[-1])
    if src and os.path.getsize(src)>0:
        dst=MODELS/rel
        dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.is_symlink() or dst.exists(): dst.unlink()
        try:
            os.symlink(src,dst)
            if local_ok(rel):
                print("CACHE hit:",rel,"<-",src); continue
        except Exception as e:
            print("symlink failed for",rel,":",e)
    miss.append(rel)

if miss:
    print("downloading",len(miss),"files from Comfy-Org/MiniMax-H3 (HF token active)")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Comfy-Org/MiniMax-H3",allow_patterns=miss,local_dir=str(MODELS))

# --- verify every file + write MANIFEST (source of truth for this session) ---
MANIFEST={}; bad=[]
for rel in EXPECT:
    p=MODELS/rel
    sz=p.stat().st_size if p.exists() else 0
    print(("OK  " if sz else "MISS"),sz,rel)
    if not sz: bad.append(rel)
    else: MANIFEST[rel]={"size":sz,"source":("kaggle_input" if p.is_symlink() else "hf")}
if bad: raise RuntimeError("model files missing after download: %s"%bad)
json.dump(MANIFEST,open(str(MODELS/"MANIFEST.json"),"w"),indent=1)
print("MANIFEST.json written:",len(MANIFEST),"files")
subprocess.run(["curl","-L","-sSf","-o","/tmp/h3_t2v.json","https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json"],check=True)
print("assets staged in /tmp/ComfyUI/models, workflow at /tmp/h3_t2v.json")


In [ ]:

# ============ PROMPT BUILDER: settings dict -> validated API prompt (shared by smoke + API) ============
import json, base64, hashlib, urllib.request, urllib.error
from pathlib import Path

BASE="http://127.0.0.1:8188"
MISSING=object()

def load_info():
    return json.loads(urllib.request.urlopen(BASE+"/object_info",timeout=60).read())

def _save_frame(durl, slot):
    """data-URL image -> /tmp/ComfyUI/input/<name>; returns the LoadImage name."""
    head,b64=durl.split(",",1)
    mime=head.split(";")[0].replace("data:","")
    ext={"image/png":"png","image/jpeg":"jpg","image/webp":"webp","image/gif":"gif"}.get(mime,"png")
    raw=base64.b64decode(b64)
    name="h3_%s_%s.%s"%(slot,hashlib.sha1(raw).hexdigest()[:10],ext)
    d=Path("/tmp/ComfyUI/input"); d.mkdir(parents=True,exist_ok=True)
    (d/name).write_bytes(raw)
    return name

def build_prompt(cfg):
    """cfg (a PRESETS-style dict) -> (prompt, meta). Flattens the official subgraph,
    applies settings, wires optional first/last frame, inserts H3MultiStream."""
    wf=json.load(open("/tmp/h3_t2v.json"))
    SUBID=(wf.get("definitions") or {}).get("subgraphs")[0]["id"]
    inst=[n for n in wf["nodes"] if n["type"]==SUBID][0]
    iw=inst.setdefault("widgets_values_named",{})
    iw.update({
        "prompt":cfg["prompt"], "value_1":float(cfg["duration"]),
        "noise_seed":int(cfg["seed"]),
        "unet_name":cfg["unet"], "clip_name":cfg["clip"],
        "vae_name":cfg["vae"], "vae_name_1":cfg["audio_vae"],
        "value":bool(cfg["turbo"]), "lora_name":cfg["lora"],
        "strength_model_1":float(cfg["turbo_strength"]),
        "value_2":int(cfg["turbo_steps"]),
    })
    for n in wf["nodes"]:
        if n["type"]=="ResolutionSelector":
            rw=n.setdefault("widgets_values_named",{})
            rw["aspect_ratio"]=cfg["aspect"]; rw["megapixels"]=cfg["megapixels"]

    # optional first/last-frame conditioning: LoadImage -> subgraph input (link ids free-form)
    frames=[]
    for slot,key in ((0,"first_frame"),(1,"last_frame")):
        durl=cfg.get(key+"_b64")
        if not durl: continue
        name=_save_frame(durl,key)
        nid=9001+slot; lid=7777+slot
        wf["nodes"].append({"id":nid,"type":"LoadImage","pos":[0,0],"size":[140,100],"flags":{},
            "order":99,"mode":0,"inputs":[],
            "outputs":[{"name":"IMAGE","type":"IMAGE","links":[lid],"slot_index":0}],
            "properties":{},"widgets_values":[name],"widgets_values_named":{"image":name}})
        for ii in inst.get("inputs") or []:
            if ii.get("name")==key: ii["link"]=lid
        wf["links"].append([lid,nid,0,inst["id"],slot,"IMAGE"])
        frames.append(name)
    if frames: print("frame conditioning:",frames)

    info=load_info()
    def schema(ct):
        if ct not in info:
            raise RuntimeError("node type not registered: "+ct)
        s=info[ct]["input"]
        return set(s.get("required",{}))|set(s.get("optional",{}))

    subs={s["id"]:s for s in ((wf.get("definitions") or {}).get("subgraphs") or [])}
    main_links={l[0]:l for l in (wf.get("links") or [])}
    instance_out={}
    prompt={}
    _next=[1000]
    def alloc():
        _next[0]+=1
        return str(_next[0])

    def main_value(link_id):
        l=main_links[link_id]
        src,slot=l[1],l[2]
        if str(src) in instance_out:
            return instance_out[str(src)](slot)
        return [str(src),slot]

    def expand_instance(node):
        sub=subs[node["type"]]
        ilinks={l["id"]:l for l in (sub.get("links") or [])}
        inst_inputs={i["name"]:i for i in (node.get("inputs") or [])}
        wnamed=node.get("widgets_values_named") or {}

        def sub_input(slot):
            name=sub["inputs"][slot]["name"]
            ii=inst_inputs.get(name)
            if ii is not None and ii.get("link") is not None:
                return main_value(ii["link"])
            if name in wnamed:
                return wnamed[name]
            return MISSING

        idmap={}
        for n in sub["nodes"]:
            schema(n["type"])
            idmap[str(n["id"])]=alloc()

        def out_value(slot):
            out=(sub.get("outputs") or [])[slot]
            lk=ilinks[out["linkIds"][0]]
            return [idmap[str(lk["origin_id"])],lk["origin_slot"]]

        for n in sub["nodes"]:
            valid=schema(n["type"])
            linked=set(); inputs={}
            for inp in (n.get("inputs") or []):
                nm=inp["name"]
                if inp.get("link") is None: continue   # links always included: autogrow slots
                linked.add(nm)                          # (values.a) only exist after finalization
                lk=ilinks.get(inp["link"])
                if lk is None: raise RuntimeError("inner link missing: %s"%inp["link"])
                if lk["origin_id"]==-10:
                    v=sub_input(lk["origin_slot"])
                else:
                    oid=idmap.get(str(lk["origin_id"]))
                    if oid is None: raise RuntimeError("inner origin missing: %s"%lk["origin_id"])
                    v=[oid,lk["origin_slot"]]
                if v is MISSING: continue
                inputs[nm]=v
            named=n.get("widgets_values_named") or {}
            for nm in valid:
                if nm in inputs or nm in linked: continue
                if nm in named: inputs[nm]=named[nm]
            prompt[idmap[str(n["id"])]]={"class_type":n["type"],"inputs":inputs}
        return out_value

    # pass 1: flatten every subgraph instance
    for n in wf.get("nodes",[]):
        if n["type"] in subs:
            instance_out[str(n["id"])]=expand_instance(n)
            print("expanded subgraph",n["type"],"-> nodes:",len(prompt))

    # pass 2: regular (UI graph) nodes
    for n in wf.get("nodes",[]):
        t=n["type"]
        if t in subs: continue
        if t not in info:
            print("skipping UI-only node:",t,"#%s"%n["id"]); continue
        valid=schema(t)
        linked=set(); inputs={}
        for inp in (n.get("inputs") or []):
            nm=inp["name"]
            if inp.get("link") is None: continue
            linked.add(nm)
            v=main_value(inp["link"])
            if v is MISSING: continue
            inputs[nm]=v
        named=n.get("widgets_values_named") or {}
        for nm in valid:
            if nm in inputs or nm in linked: continue
            if nm in named: inputs[nm]=named[nm]
        prompt[str(n["id"])]={"class_type":t,"inputs":inputs}

    print("flattened prompt nodes:",len(prompt))

    # turbo sanity: one boolean must drive both switches (LoRA branch + step count)
    if cfg["turbo"]:
        pb=[nid for nid,n in prompt.items()
            if n["class_type"]=="PrimitiveBoolean" and n["inputs"].get("value") is True]
        if not pb: raise RuntimeError("turbo enabled but no PrimitiveBoolean(value=True) in prompt")
        print("turbo path armed: PrimitiveBoolean",pb,"-> steps",cfg["turbo_steps"],"+ LoRA")
    else:
        print("dense path: 20 steps, no LoRA")

    # insert H3MultiStream after the model switch, feeding BasicGuider + BasicScheduler
    schema("H3MultiStream")
    targets=[]
    for nid,node in prompt.items():
        if node["class_type"] in ("BasicGuider","BasicScheduler") and isinstance(node["inputs"].get("model"),list):
            targets.append((nid,node["inputs"]["model"]))
    if not targets:
        raise RuntimeError("no BasicGuider/BasicScheduler with a linked model input")
    src=targets[0][1]
    h3_in={"model":src,"enabled":True,"second_gpu":-1,"exchange":"host","exchange_chunks":8,
           "sparse_attention":False,"dynamic_vram":"keep","vram_block_cache":False,
           "vram_reserve_gb":2.0,"sparse_vsa":False,"weight_cache":False,"cache_ram_reserve_gb":0.0}
    valid=schema("H3MultiStream")
    missing=[k for k in valid if k not in h3_in and k!="gpus" and not isinstance(h3_in.get(k),list)]
    h3_in={k:v for k,v in h3_in.items() if k in valid}
    prompt["9999"]={"class_type":"H3MultiStream","inputs":h3_in}
    n_p=0
    for nid,v in targets:
        if v==src:
            prompt[nid]["inputs"]["model"]=["9999",0]; n_p+=1
    print("H3MultiStream inserted as node 9999, feeding",n_p,"consumers; model source",src)
    if missing: print("WARNING: H3 inputs not supplied:",missing)

    steps=cfg["turbo_steps"] if cfg["turbo"] else 20
    meta={"nodes":len(prompt),"turbo":bool(cfg["turbo"]),"steps":steps,
          "megapixels":cfg["megapixels"],"aspect":cfg["aspect"],
          "duration":cfg["duration"],"frames_injected":frames}
    return prompt, meta


In [ ]:

import subprocess,sys,os,time,json
c=subprocess.Popen([sys.executable,"/tmp/ComfyUI/main.py","--listen","127.0.0.1","--port","8188","--preview-method","latent2rgb"],
                   cwd="/tmp/ComfyUI", stdout=open("/tmp/comfy.log","w"), stderr=subprocess.STDOUT)
print("Comfy pid",c.pid)
# sample both GPUs every 10s -> /tmp/gpus.log (dual-GPU evidence for the smoke test)
try:
    subprocess.Popen(["nvidia-smi","--query-gpu=index,utilization.gpu,utilization.memory,memory.used,power.draw",
                      "--format=csv,noheader","-l","10"],
                     stdout=open("/tmp/gpus.log","w"), stderr=subprocess.STDOUT)
except Exception as e:
    print("gpu monitor not started:",e)
ready=False
for i in range(72):
    try:
        import urllib.request as ur
        ur.urlopen("http://127.0.0.1:8188/history", timeout=3)
        ready=True; break
    except Exception: time.sleep(5)
if not ready:
    raise RuntimeError("ComfyUI did not listen in time")
print("ComfyUI local ready")


In [ ]:

# ============ HUB API + PUBLIC PROXY: /h3api/* served here, everything else piped to ComfyUI ============
import json, os, socket, threading, time, urllib.request, urllib.error, http.server, subprocess, shutil
from pathlib import Path
from urllib.parse import urlparse, parse_qs, quote

API_PORT=8190
COMFY=("127.0.0.1",8188)
MODELS=Path("/tmp/ComfyUI/models")
API_UP=False
UP={"since":time.time()}
DL={"running":False,"last":None}
LOCK=threading.Lock()

def _j(code,obj):
    return code,{"Content-Type":"application/json"},json.dumps(obj).encode()

def comfy(method,path,data=None,timeout=90):
    req=urllib.request.Request("http://%s:%d%s"%(COMFY[0],COMFY[1],path),
        data=(json.dumps(data).encode() if data is not None else None),method=method)
    with urllib.request.urlopen(req,timeout=timeout) as r:
        raw=r.read()
    return json.loads(raw) if raw else {}

def _inventory():
    out={}
    if not MODELS.is_dir(): return out
    for p in sorted(MODELS.rglob("*")):
        if not (p.is_file() or p.is_symlink()): continue
        try: sz=p.stat().st_size
        except OSError: continue
        rel=str(p.relative_to(MODELS))
        out[rel]={"size":sz,"source":("kaggle_input" if p.is_symlink() else "local")}
    return out

def _required():
    return ["%s/%s"%(c,n) for c,n in required_files(ACTIVE)]

# ---------------- handlers ----------------
def H_index(o):
    return _j(200,{"ok":True,"name":"MiniMax H3 generation hub","docs":"/h3api/docs",
        "endpoints":sorted(["%s %s"%(m,p) for (m,p) in ROUTES]),
        "comfy_root":"/","native_prompt_api":"/prompt"})

def H_health(o):
    try: comfy("GET","/system_stats",timeout=4); up=True
    except Exception: up=False
    return _j(200,{"ok":True,"comfy":up,"api_uptime_s":round(time.time()-UP["since"]),
        "active_preset":ACTIVE_PRESET,"active":ACTIVE,"required":_required()})

def H_catalog(o):
    q=parse_qs(urlparse(o["path"]).query)
    if (q.get("refresh") or ["0"])[0] in ("1","true"):
        try:
            CORE=[{"path":f["path"],"size":int(f.get("size") or 0)} for f in
                  hf_get("https://huggingface.co/api/models/%s/tree/main?recursive=true"%CORE_REPO)
                  if f.get("type")=="file" and not f["path"].startswith(".")]
            COMM=[{"id":m["id"],"downloads":int(m.get("downloads") or 0)} for m in
                  hf_get("https://huggingface.co/api/models?search=MiniMax-H3&limit=100&sort=downloads&direction=-1")]
            reg=json.load(open("/tmp/h3_registry.json")); reg["core"]=CORE; reg["community"]=COMM
            json.dump(reg,open("/tmp/h3_registry.json","w"))
        except Exception as e:
            return _j(502,{"ok":False,"error":"live refresh failed: %s"%e})
    try: reg=json.load(open("/tmp/h3_registry.json"))
    except Exception as e: return _j(503,{"ok":False,"error":"registry not built yet: %s"%e})
    cache=reg.get("cache",{})
    for f in reg.get("core",[]):
        p=MODELS/f["path"]
        f["status"]="local" if (p.exists() and p.stat().st_size>0) else             ("cached" if f["path"].rsplit("/",1)[-1] in cache else "remote")
    return _j(200,{"ok":True,"repo":reg.get("repo"),"files":reg.get("core",[]),
        "community":reg.get("community",[]),"cache_files":len(cache)})

def H_models(o):
    mf={}
    try: mf=json.load(open(str(MODELS/"MANIFEST.json")))
    except Exception: pass
    return _j(200,{"ok":True,"files":_inventory(),"manifest":mf,"required":_required(),
        "active_preset":ACTIVE_PRESET})

def _apply_settings(b):
    global ACTIVE, ACTIVE_PRESET
    if b.get("preset") is not None:
        if b["preset"] not in PRESETS:
            return _j(400,{"ok":False,"error":"unknown preset","available":sorted(PRESETS)})
        ACTIVE_PRESET=b["preset"]; ACTIVE=dict(PRESETS[ACTIVE_PRESET])
    if isinstance(b.get("set"),dict):
        ACTIVE.update(b["set"]); ACTIVE_PRESET="(custom)"
    return None

def H_settings(o):
    if o["method"]=="GET":
        return _j(200,{"ok":True,"active_preset":ACTIVE_PRESET,"presets":PRESETS,"active":ACTIVE,
            "required":_required(),
            "missing":[r for r in _required() if not (MODELS/r).exists() or (MODELS/r).stat().st_size==0],
            "note":"H3 has no negative prompt (CFG-distilled): prompt + optional first/last frame only"})
    err=_apply_settings(o["body"])
    if err: return err
    return _j(200,{"ok":True,"active_preset":ACTIVE_PRESET,"active":ACTIVE,"required":_required(),
        "missing":[r for r in _required() if not (MODELS/r).exists() or (MODELS/r).stat().st_size==0]})

def _start_download(files,note):
    def run():
        DL["running"]=True
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=CORE_REPO,allow_patterns=files,
                              local_dir=str(MODELS),token=os.environ.get("HF_TOKEN"))
            DL["last"]={"ok":True,"files":files,"note":note,"t":time.time()}
        except Exception as e:
            DL["last"]={"ok":False,"error":str(e)[:500],"files":files,"t":time.time()}
        finally:
            DL["running"]=False
    threading.Thread(target=run,daemon=True).start()

def H_download(o):
    b=o["body"] or {}
    files=b.get("files") or ([b["file"]] if b.get("file") else [])
    files=[f for f in files if isinstance(f,str) and f and not f.startswith("/") and ".." not in f]
    if not files:
        return _j(400,{"ok":False,"error":"pass {'file':'loras/x.safetensors'} or {'files':[...]}"})
    with LOCK:
        if DL["running"]: return _j(409,{"ok":False,"error":"a download is already running","last":DL["last"]})
        _start_download(files,"manual")
    return _j(202,{"ok":True,"queued":files,"status":"/h3api/download"})

def H_download_status(o):
    return _j(200,{"ok":True,"running":DL["running"],"last":DL["last"]})

def H_select(o):
    """switch model/settings: downloads the new set (async) and deletes superseded local
    files so disk and GPU stay clean — the previous model never lingers."""
    b=o["body"] or {}
    err=_apply_settings(b)
    if err: return err
    keep=set(_required())
    PRUNE_DIRS={"diffusion_models","text_encoders","vae","loras"}
    freed=0; pruned=[]
    if b.get("prune",True):
        for rel in list(_inventory()):
            if rel=="MANIFEST.json" or rel in keep: continue
            if not rel.endswith(".safetensors"): continue
            if rel.split("/")[0] not in PRUNE_DIRS: continue
            p=MODELS/rel
            try:
                sz=p.stat().st_size; p.unlink(); freed+=sz; pruned.append(rel)
            except Exception: pass
    missing=sorted(r for r in keep if not (MODELS/r).exists() or (MODELS/r).stat().st_size==0)
    queued=False
    if missing:
        with LOCK:
            if not DL["running"]:
                _start_download(missing,"select"); queued=True
        if not queued:
            return _j(409,{"ok":True,"error":"download busy; retry later","active_preset":ACTIVE_PRESET,
                "active":ACTIVE,"pruned":pruned,"freed_bytes":freed,"missing":missing,"queued":False})
    return _j(200 if not missing else 202,{"ok":True,"active_preset":ACTIVE_PRESET,"active":ACTIVE,
        "pruned":pruned,"freed_bytes":freed,"missing":missing,"queued":queued,
        "status":"/h3api/download"})

def H_generate(o):
    b=o["body"] or {}
    cfg=dict(ACTIVE)
    if b.get("preset") is not None:
        if b["preset"] not in PRESETS:
            return _j(400,{"ok":False,"error":"unknown preset","available":sorted(PRESETS)})
        cfg=dict(PRESETS[b["preset"]])
    if isinstance(b.get("set"),dict): cfg.update(b["set"])
    if b.get("prompt"): cfg["prompt"]=str(b["prompt"])[:6000]
    if b.get("first_frame"): cfg["first_frame_b64"]=b["first_frame"]
    if b.get("last_frame"): cfg["last_frame_b64"]=b["last_frame"]
    if b.get("seed") is not None:
        try: cfg["seed"]=int(b["seed"])
        except Exception: return _j(400,{"ok":False,"error":"seed must be an integer"})
    try:
        prompt,meta=build_prompt(cfg)
    except Exception as e:
        return _j(500,{"ok":False,"error":"prompt build failed: %s"%e})
    try:
        res=comfy("POST","/prompt",{"prompt":prompt,"extra_data":{"preview_method":"latent2rgb"}},timeout=60)
    except urllib.error.HTTPError as e:
        return _j(502,{"ok":False,"error":"comfy rejected the prompt","detail":e.read().decode()[:4000]})
    except Exception as e:
        return _j(502,{"ok":False,"error":"comfy unreachable: %s"%e})
    return _j(202,{"ok":True,"prompt_id":res.get("prompt_id"),"meta":meta,
        "poll":"/h3api/jobs","history":"/history/"+str(res.get("prompt_id"))})

def H_jobs(o):
    try:
        hist=comfy("GET","/history")
        qq=comfy("GET","/queue")
    except Exception as e:
        return _j(503,{"ok":False,"error":"comfy unreachable: %s"%e})
    jobs=[]
    for pid,entry in list(hist.items())[-25:]:
        st=entry.get("status",{})
        jobs.append({"id":pid,"status":st.get("status_str"),
            "completed":st.get("completed"),"remaining":st.get("remaining"),
            "outputs":list((entry.get("outputs") or {}).keys())})
    jobs.reverse()
    return _j(200,{"ok":True,
        "queue":{"running":len(qq.get("queue_running") or []),
                 "pending":len(qq.get("queue_pending") or [])},
        "jobs":jobs})

def H_outputs(o):
    d=Path("/tmp/ComfyUI/output"); out=[]
    if d.is_dir():
        for p in d.rglob("*"):
            if not p.is_file(): continue
            try: st=p.stat()
            except OSError: continue
            out.append({"name":p.name,"size":st.st_size,"modified":int(st.st_mtime),
                "url":"/view?filename=%s&subfolder=&type=output"%quote(p.name)})
    out.sort(key=lambda x:-x["modified"])
    return _j(200,{"ok":True,"total":len(out),"files":out[:200]})

def H_free(o):
    try: comfy("POST","/free",{"unload_models":True,"free_memory":True})
    except Exception as e: return _j(502,{"ok":False,"error":"comfy unreachable: %s"%e})
    return _j(200,{"ok":True,"unloaded":True,"note":"models freed from GPU; next generate reloads"})

def H_gpus(o):
    try:
        r=subprocess.run(["nvidia-smi","--query-gpu=index,name,utilization.gpu,memory.used,power.draw",
                          "--format=csv,noheader"],capture_output=True,text=True,timeout=15)
        lines=[l.strip() for l in (r.stdout or "").splitlines() if l.strip()]
        return _j(200,{"ok":bool(lines),"gpus":lines or None,
                       "detail":(r.stderr or "").strip()[:200] or None})
    except Exception as e:
        return _j(200,{"ok":False,"gpus":None,"detail":str(e)[:200]})

def H_import(o):
    """POST /h3api/import?path=<category>/<file.safetensors> with raw file bytes as body."""
    rel=(parse_qs(urlparse(o["path"]).query).get("path") or [""])[0].strip()
    if not rel or ".." in rel or rel.startswith("/") or not rel.endswith(".safetensors"):
        return _j(400,{"ok":False,"error":"?path=<category>/<file.safetensors> required"})
    cl=int(o["headers"].get("Content-Length") or 0)
    if cl<=0: return _j(400,{"ok":False,"error":"raw file bytes required as body"})
    free=shutil.disk_usage("/tmp").free
    if cl>free*0.95:
        return _j(507,{"ok":False,"error":"not enough disk for import","free":free,"need":cl})
    dst=MODELS/rel; dst.parent.mkdir(parents=True,exist_ok=True)
    n=0
    with open(dst,"wb") as f:
        left=cl
        while left>0:
            chunk=o["rfile"](min(1<<20,left))
            if not chunk: break
            f.write(chunk); n+=len(chunk); left-=len(chunk)
    if n!=cl:
        try: dst.unlink()
        except Exception: pass
        return _j(400,{"ok":False,"error":"incomplete upload (%d/%d bytes)"%(n,cl)})
    return _j(201,{"ok":True,"path":rel,"size":n,
        "note":"file is on disk; use /h3api/select or /h3api/generate to use it"})

DOCS_HTML = """<!doctype html>
<html><head><meta charset="utf-8"><title>MiniMax H3 Hub API</title>
<style>
body{font-family:ui-sans-serif,system-ui,sans-serif;margin:2rem auto;max-width:60rem;line-height:1.5;color:#111}
code,pre{background:#f4f4f5;border-radius:4px;padding:.1rem .3rem;font-family:ui-monospace,monospace}
pre{padding:.8rem;overflow-x:auto} table{border-collapse:collapse;width:100%}
th,td{border:1px solid #ddd;padding:.45rem;text-align:left;vertical-align:top} th{background:#fafafa}
h1{font-size:1.4rem} h2{font-size:1.05rem;margin-top:1.8rem}</style></head><body>
<h1>MiniMax H3 generation hub &mdash; API reference</h1>
<p>One public Base URL serves three things: the <a href="/">ComfyUI web UI</a>,
the native Comfy API (<code>POST /prompt</code>, <code>GET /history</code>, <code>GET /ws</code>),
and this management hub under <code>/h3api/*</code>. All hub responses are JSON:
<code>{"ok": true, ...}</code> or <code>{"ok": false, "error": "..."}</code>.</p>

<h2>Endpoints</h2>
<table><tr><th>Method &amp; path</th><th>Purpose</th></tr>
<tr><td><code>GET /h3api/</code></td><td>Endpoint index</td></tr>
<tr><td><code>GET /h3api/docs</code></td><td>This page</td></tr>
<tr><td><code>GET /h3api/health</code></td><td>API + Comfy liveness, uptime, active preset</td></tr>
<tr><td><code>GET /h3api/catalog</code></td><td>Live HF model catalog (39 core files + community repos)
 with <code>local/cached/remote</code> status. <code>?refresh=1</code> re-queries Hugging Face now</td></tr>
<tr><td><code>GET /h3api/models</code></td><td>On-disk models, sizes, sources, MANIFEST, required set</td></tr>
<tr><td><code>GET /h3api/settings</code></td><td>Presets + active generation settings</td></tr>
<tr><td><code>POST /h3api/settings</code></td><td><code>{"preset":"fast_smoke"}</code> and/or
 <code>{"set":{"megapixels":0.4,"duration":5,"seed":1,"turbo":true,"turbo_steps":4,"aspect":"16:9 (Widescreen)"}}</code></td></tr>
<tr><td><code>POST /h3api/select</code></td><td>Switch model/settings: downloads the new set (async),
 <b>deletes superseded local files</b> (<code>{"prune":false}</code> to keep them)</td></tr>
<tr><td><code>POST /h3api/download</code></td><td>Download files now:
 <code>{"file":"loras/x.safetensors"}</code> or <code>{"files":[...]}</code> from Comfy-Org/MiniMax-H3</td></tr>
<tr><td><code>GET /h3api/download</code></td><td>Download status (running / last result)</td></tr>
<tr><td><code>POST /h3api/generate</code></td><td>Queue a generation (see below)</td></tr>
<tr><td><code>GET /h3api/jobs</code></td><td>Queue + history status of recent jobs</td></tr>
<tr><td><code>GET /h3api/outputs</code></td><td>Generated files + <code>/view</code> URLs</td></tr>
<tr><td><code>POST /h3api/free</code></td><td>Unload models from GPU / free VRAM</td></tr>
<tr><td><code>GET /h3api/gpus</code></td><td>nvidia-smi snapshot (index, util, VRAM, power)</td></tr>
<tr><td><code>POST /h3api/import</code></td><td>Upload a model file:
 <code>?path=loras/file.safetensors</code>, raw bytes as body</td></tr>
<tr><td><code>POST /prompt</code></td><td>Native Comfy API (pass a fully built API prompt)</td></tr></table>

<h2>Generate</h2>
<pre>curl -X POST "$BASE/h3api/generate" -H 'Content-Type: application/json' -d '{
  "prompt": "A calm ocean wave rolling onto a quiet beach at golden hour",
  "seed": 12345,
  "set": {"megapixels": 0.4, "duration": 5}
}'</pre>
<p>Returns <code>{"ok":true,"prompt_id":"...","meta":{...}}</code>. Optional keys:
<code>preset</code> ("fast_smoke" | "quality"), <code>set</code> (any settings keys),
<code>first_frame</code> / <code>last_frame</code> as <code>data:image/png;base64,...</code>
URLs (image-to-video / last-frame conditioning), <code>seed</code>.
Poll <code>GET /h3api/jobs</code> until the job status is <code>success</code>.</p>

<h2>Settings surface</h2>
<table><tr><th>Key</th><th>Meaning</th></tr>
<tr><td><code>prompt</code></td><td>Text prompt (H3 is CFG-distilled: there is <b>no negative prompt</b>)</td></tr>
<tr><td><code>aspect</code>, <code>megapixels</code></td><td>Aspect ratio + quality: 0.4 &rarr; 864&times;480,
 0.98 &rarr; 1344&times;768 (16:9)</td></tr>
<tr><td><code>duration</code></td><td>Seconds at 24 fps (trained range 5&ndash;15 s)</td></tr>
<tr><td><code>steps</code>, <code>turbo</code>, <code>turbo_steps</code>, <code>lora</code>,
<code>turbo_strength</code></td><td>Sampling: dense 20-step, or turbo LoRA (4-step fast path)</td></tr>
<tr><td><code>seed</code></td><td>Noise seed</td></tr>
<tr><td><code>unet</code>, <code>clip</code>, <code>vae</code>, <code>audio_vae</code></td><td>Model files
(select via <code>/h3api/select</code>; switching downloads the new set and deletes the previous one)</td></tr>
<tr><td><code>first_frame_b64</code>, <code>last_frame_b64</code></td><td>Frame conditioning (data URLs)</td></tr>
</table>

<h2>Model lifecycle</h2>
<p>Boot: registry queries Hugging Face live &rarr; catalog lists every file with
<code>local/cached/remote</code> &rarr; assets mirror the Kaggle Dataset cache under
<code>/kaggle/input</code> (symlinks) &rarr; only missing files download (HF token).
A dataset write-back (200 GB cap, auto-skip) backs up downloaded models for the next boot.</p>
</body></html>
"""

ROUTES={
    ("GET","/h3api"):H_index, ("GET","/h3api/"):H_index,
    ("GET","/h3api/health"):H_health, ("GET","/h3api/catalog"):H_catalog,
    ("GET","/h3api/models"):H_models,
    ("GET","/h3api/settings"):H_settings, ("POST","/h3api/settings"):H_settings,
    ("GET","/h3api/download"):H_download_status, ("POST","/h3api/download"):H_download,
    ("POST","/h3api/select"):H_select, ("POST","/h3api/generate"):H_generate,
    ("GET","/h3api/jobs"):H_jobs, ("GET","/h3api/outputs"):H_outputs,
    ("POST","/h3api/free"):H_free, ("GET","/h3api/gpus"):H_gpus,
    ("POST","/h3api/import"):H_import,
}

class H(http.server.BaseHTTPRequestHandler):
    protocol_version="HTTP/1.1"
    server_version="H3Hub/1.0"

    def log_message(self,*a): pass   # keep notebook output clean

    def _send(self,code,headers,body):
        try:
            self.send_response(code)
            for k,v in headers.items(): self.send_header(k,v)
            self.send_header("Access-Control-Allow-Origin","*")
            self.send_header("Access-Control-Allow-Headers","Content-Type")
            self.send_header("Access-Control-Allow-Methods","GET,POST,OPTIONS")
            self.send_header("Content-Length",str(len(body)))
            self.end_headers()
            self.wfile.write(body)
        except Exception: pass

    def _read_body(self):
        te=(self.headers.get("Transfer-Encoding") or "").lower()
        if "chunked" in te:
            buf=b""
            while True:
                line=self.rfile.readline().strip()
                if b";" in line: line=line.split(b";")[0]
                try: n=int(line or b"0",16)
                except ValueError: break
                if n==0:
                    while True:
                        t=self.rfile.readline()
                        if t in (b"\r\n",b"\n",b""): break
                    break
                buf+=self.rfile.read(n); self.rfile.readline()
            return buf
        cl=int(self.headers.get("Content-Length") or 0)
        return self.rfile.read(cl) if cl>0 else b""

    def do_OPTIONS(self): self._send(204,{"Content-Type":"text/plain"},b"")
    def do_GET(self): self._route("GET")
    def do_POST(self): self._route("POST")

    def _route(self,method):
        path=self.path.split("?")[0]
        if method=="POST" and path=="/h3api/import":
            try:
                self._send(*H_import({"path":self.path,"headers":self.headers,
                                      "rfile":self.rfile.read}))
            except Exception as e:
                self._send(*_j(500,{"ok":False,"error":str(e)}))
            return
        fn=ROUTES.get((method,path))
        if fn is None and path.startswith("/h3api"):
            self._send(*_j(404,{"ok":False,"error":"unknown endpoint","docs":"/h3api/docs"}))
            return
        try:
            body=self._read_body()
            if fn is not None:
                obj={}
                if body:
                    try: obj=json.loads(body)
                    except Exception: obj={"_raw":len(body)}
                self._send(*fn({"method":method,"body":obj,"headers":self.headers,"path":self.path}))
            else:
                self._proxy(method,body)
        except Exception as e:
            import traceback; traceback.print_exc()
            self._send(*_j(500,{"ok":False,"error":str(e)}))

    def _proxy(self,method,body):
        if (self.headers.get("Upgrade") or "").lower()=="websocket":
            return self._ws()
        req=urllib.request.Request("http://%s:%d%s"%(COMFY[0],COMFY[1],self.path),
            data=(body if body and method!="GET" else None),method=method)
        for k,v in self.headers.items():
            lk=k.lower()
            if lk in ("host","connection","upgrade","content-length",
                      "accept-encoding","transfer-encoding","keep-alive"): continue
            req.add_header(k,v)
        if body: req.add_header("Content-Length",str(len(body)))
        try:
            with urllib.request.urlopen(req,timeout=600) as r:
                data=r.read(); st=r.status; rh=dict(r.headers)
        except urllib.error.HTTPError as e:
            data=e.read(); st=e.code; rh=dict(e.headers)
        except Exception as e:
            self._send(*_j(502,{"ok":False,"error":"comfy proxy: %s"%e})); return
        try:
            self.send_response(st)
            for k,v in rh.items():
                if k.lower() in ("transfer-encoding","connection","content-length",
                                 "content-encoding","keep-alive"): continue
                self.send_header(k,v)
            self.send_header("Content-Length",str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except Exception: pass

    def _ws(self):
        """WebSocket passthrough: raw bidirectional pipe browser <-> ComfyUI."""
        try:
            up=socket.create_connection(COMFY,timeout=10)
        except Exception as e:
            self._send(*_j(502,{"ok":False,"error":"comfy ws unreachable: %s"%e})); return
        head="GET %s HTTP/1.1\r\n"%self.path
        for k,v in self.headers.items():
            if k.lower()=="host": v="%s:%d"%COMFY
            head+="%s: %s\r\n"%(k,v)
        head+="\r\n"
        up.sendall(head.encode())
        src=self.connection
        self.close_connection=True
        def pump(a,b):
            try:
                while True:
                    d=a.recv(65536)
                    if not d: break
                    b.sendall(d)
            except Exception: pass
            try: b.shutdown(socket.SHUT_WR)
            except Exception: pass
        threading.Thread(target=pump,args=(src,up),daemon=True).start()
        pump(up,src)
        try: up.close()
        except Exception: pass

def H_docs(o):
    return 200,{"Content-Type":"text/html; charset=utf-8"},DOCS_HTML.encode()

ROUTES[("GET","/h3api/docs")]=H_docs

# ---------------- start ----------------
if os.environ.get("H3_API","1")!="0":
    try:
        srv=http.server.ThreadingHTTPServer(("127.0.0.1",API_PORT),H)
        srv.daemon_threads=True
        threading.Thread(target=srv.serve_forever,daemon=True).start()
        API_UP=True
        print("hub API + proxy listening on http://127.0.0.1:%d (docs: /h3api/docs)"%API_PORT)
    except Exception as e:
        API_UP=False
        print("hub API failed to start (continuing without it):",e)
else:
    API_UP=False
    print("hub API disabled (H3_API=0)")


In [ ]:

import json,time,urllib.request,urllib.error,os
prompt, meta = build_prompt(ACTIVE)          # shared builder (convert cell)
print("build meta:", json.dumps(meta))
body={"prompt":prompt,"extra_data":{"preview_method":"latent2rgb"}}
try:
    r=urllib.request.Request(BASE+"/prompt",data=json.dumps(body).encode(),headers={"Content-Type":"application/json"})
    resp=urllib.request.urlopen(r,timeout=60).read().decode()
    print("submitted:",resp)
except urllib.error.HTTPError as ee:
    print("prompt failed",ee,"body:",ee.read().decode()[:6000])
    raise


In [ ]:

import json,time,urllib.request,os
pid_j=json.loads(resp)
pid=pid_j.get("prompt_id") or pid_j.get("id") or pid_j.get("prompt")
print("prompt_id",pid)
got=None
seen=set()
KEY=("[GPUs]","[MultiStream]","[VAE split]","[Cache]","UNSPLIT","unsplit on 1 GPU")
for i in range(1080):                      # 3 h: fast smoke = 4 steps @ 0.4 MP (~40-60 min est.)
    try:
        h=json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}",timeout=5).read())
        if h and h.get(pid):
            got=h.get(pid); break
    except Exception: pass
    if i%12==0:
        try:
            lines=[l for l in open("/tmp/comfy.log",errors="ignore").read().splitlines() if l.strip()]
            print(f"[{i*10}s]", lines[-1][:200] if lines else "(no log yet)")
            diag=[l for l in lines if any(k in l for k in KEY)]
            new=[l for l in diag if l not in seen]
            for l in new: print("   [h3ms]", l[:250])       # [GPUs] plan, active ranks, step times
            seen.update(new)
            if any(("UNSPLIT" in l or "unsplit on 1 GPU" in l) for l in new):
                raise RuntimeError("single-GPU fallback detected: "+new[-1][:250])
        except RuntimeError: raise
        except Exception: pass
        try:   # live per-GPU util: index, util.gpu, util.mem, mem.used, power
            g=[l for l in open("/tmp/gpus.log",errors="ignore").read().splitlines() if l.strip()][-2:]
            for gl in g: print("   [gpu]", gl[:160])
        except Exception: pass
    time.sleep(10)
if not got: raise RuntimeError("smoke did not finish within 3 h")
# ---- dual-GPU verification: both T4s must actually be in the split ----
cl=open("/tmp/comfy.log",errors="ignore").read()
ev=[l for l in cl.splitlines() if ("[GPUs]" in l or "[MultiStream] active" in l or "UNSPLIT" in l or "unsplit on 1 GPU" in l)]
for l in ev: print("h3ms:",l[:300])
if "UNSPLIT" in cl or "running unsplit on 1 GPU" in cl:
    raise RuntimeError("single-GPU fallback detected -- smoke did NOT use both T4s")
if not any("active: 2 ranks" in l for l in ev):
    raise RuntimeError("no '[MultiStream] active: 2 ranks' evidence in comfy.log -- cannot confirm dual-GPU")
print("DUAL-GPU CONFIRMED: active: 2 ranks")
try:
    lastg=[l for l in open("/tmp/gpus.log",errors="ignore").read().splitlines() if l.strip()][-1:]
    print("last gpu sample:", lastg[0][:160] if lastg else "(none)")
except Exception: pass
st=got.get("status",{})
print("smoke status:",st.get("status_str"),"completed:",st.get("completed"),"/",st.get("remaining"))
if st.get("status_str")=="error": raise RuntimeError("smoke run had execution errors: "+json.dumps(st)[:2000])
print("smoke finished; outputs:")
for root,_,files in os.walk("/tmp/ComfyUI/output"):
    for f in files:
        if f.lower().endswith((".mp4",".webm",".mov",".webp",".gif",".png")):
            print("found:",os.path.join(root,f))


In [ ]:

import subprocess,time,re,urllib.request,json,os
from pathlib import Path
Path("/tmp/h3_api").mkdir(parents=True,exist_ok=True)
ok=False
for u in ["https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
          "https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64"]:
    try:
        data=urllib.request.urlopen(u,timeout=120).read()
        if len(data)>1000000:
            open("/tmp/h3_api/cloudflared","wb").write(data); ok=True; break
        print("cf dl too small from",u,len(data))
    except Exception as e:
        print("cf dl failed:",u,e)
if not ok: raise RuntimeError("cloudflared download failed")
os.chmod("/tmp/h3_api/cloudflared",0o755)
target = API_PORT if API_UP else 8188
print("tunnel target: http://127.0.0.1:%d %s"%(target,"(hub API + proxy)" if API_UP else "(direct Comfy)"))
log=open("/tmp/h3_api/cf.log","w")
proc=subprocess.Popen(["/tmp/h3_api/cloudflared","tunnel","--url","http://127.0.0.1:%d"%target],
                      stdout=log,stderr=subprocess.STDOUT)
url=None; pat=re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
for i in range(90):
    t=open("/tmp/h3_api/cf.log","r",encoding="utf-8",errors="ignore").read()
    m=pat.search(t)
    if m:
        try:
            r1=urllib.request.urlopen(m.group(0)+"/history",timeout=5)
            r2=urllib.request.urlopen(m.group(0)+"/h3api/health",timeout=5) if API_UP else None
            if r1.status==200 and (r2 is None or r2.status==200):
                url=m.group(0); break
        except Exception: pass
    time.sleep(2)
if not url: raise RuntimeError("cloudflare tunnel not healthy")
print("Verified public endpoint:",url)
print("\n================ MINIMAX H3 — GENERATION READY ================")
print("Base URL:"); print("  "+url)
print("ComfyUI web UI:"); print("  "+url+"/")
print("Native Comfy API:"); print("  POST "+url+"/prompt")
if API_UP:
    print("Hub API docs:"); print("  "+url+"/h3api/docs")
    print("Hub health:"); print("  "+url+"/h3api/health")
    print("Hub generate:"); print("  POST "+url+"/h3api/generate   {\"prompt\":\"...\"}")
    print("Hub settings/catalog/jobs/outputs/gpus:")
    print("  GET  "+url+"/h3api/settings | /h3api/catalog | /h3api/jobs | /h3api/outputs | /h3api/gpus")
else:
    print("(hub API unavailable this run - direct Comfy only)")
print("===============================================================")
if API_UP:
    print("live endpoint checks through the tunnel:")
    for p in ["/h3api/health","/h3api/catalog","/h3api/settings","/h3api/jobs",
              "/h3api/outputs","/h3api/gpus","/h3api/docs","/object_info","/h3api/nope"]:
        try:
            r=urllib.request.urlopen(url+p,timeout=20)
            body=r.read(150).decode("utf-8","ignore").replace("\n"," ")
            print("  %d %-18s %s"%(r.status,p,body[:110]))
        except Exception as e:
            code=getattr(e,"code",None) or type(e).__name__
            body=""
            try: body=e.read(150).decode("utf-8","ignore").replace("\n"," ")
            except Exception: pass
            print("  %s %-18s %s"%(code,p,body[:110]))
    try:
        req=urllib.request.Request(url+"/h3api/settings",data=b"{}",
              headers={"Content-Type":"application/json"},method="POST")
        r=urllib.request.urlopen(req,timeout=20)
        print("  %d POST /h3api/settings    %s"%(r.status,r.read(150).decode("utf-8","ignore")[:110]))
    except Exception as e:
        print("  POST /h3api/settings failed:",getattr(e,"code",type(e).__name__))


In [ ]:

# ===== Kaggle Dataset auto-cache write-back: 200 GB cap auto-skip, NEVER fails the run =====
# H3_CACHE_UPLOAD=auto (default: create once, refresh only on manifest drift)
#                |always|dry|never
# Uploads via kagglehub (native in-notebook token auth); classic kaggle CLI as fallback.
import json, os, subprocess, shutil, time
from pathlib import Path
POLICY=os.environ.get("H3_CACHE_UPLOAD","auto").lower()
CAP=200*10**9
TITLE="MiniMax H3 model cache"
SLUG="minimax-h3-model-cache"
KNOWN_OWNER="adityahalde8777"   # injected at build time from kernel-metadata id (last-resort owner)
MODELS=Path("/tmp/ComfyUI/models")
STAGE=Path("/tmp/h3_cache_stage")

def _kg(args,timeout=7200):
    r=subprocess.run(["kaggle"]+args,capture_output=True,text=True,timeout=timeout,
                     stdin=subprocess.DEVNULL)
    return r.returncode,(r.stdout or "")+"\n"+(r.stderr or "")

# owner detection: modern Kaggle notebooks auth via a TOKEN FILE (kagglehub's native
# notebook auth), not KAGGLE_USERNAME/kaggle.json - v15 proved the old check fails.
def _username():
    try:
        import kagglehub
        u=(kagglehub.whoami(verbose=False) or {}).get("username")
        if u: return u,"kagglehub.whoami"
    except Exception as e:
        print("auth: kagglehub.whoami unavailable (%s)"%type(e).__name__)
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return os.environ["KAGGLE_USERNAME"],"env KAGGLE_USERNAME/KAGGLE_KEY"
    cands=[os.path.expanduser("~/.kaggle/kaggle.json")]
    if os.environ.get("KAGGLE_CONFIG_DIR"):
        cands.append(os.path.join(os.environ["KAGGLE_CONFIG_DIR"],"kaggle.json"))
    cands+=["/root/.kaggle/kaggle.json","/home/kaggle/.kaggle/kaggle.json",
            "/kaggle/.kaggle/kaggle.json"]
    for c in cands:
        try:
            u=json.load(open(c)).get("username")
            if u: return u,"kaggle.json (%s)"%c
        except Exception: pass
    return KNOWN_OWNER,"embedded kernel owner"

def _remote_manifest(ref):
    # returns (manifest|None, status) - kagglehub first (native notebook auth)
    shutil.rmtree("/tmp/h3_remote_chk",ignore_errors=True)
    try:
        import kagglehub
        p=kagglehub.dataset_download(ref,path="MANIFEST.json",output_dir="/tmp/h3_remote_chk")
        try: return json.load(open(p)),"kagglehub probe ok"
        except Exception as e: return None,"kagglehub: manifest unreadable (%s)"%type(e).__name__
    except Exception as e:
        kh="kagglehub probe: %s"%type(e).__name__
    rc,out=_kg(["datasets","download",ref,"-f","MANIFEST.json","-p","/tmp/h3_remote_chk"],timeout=300)
    if rc==0:
        try: return json.load(open("/tmp/h3_remote_chk/MANIFEST.json")),"cli probe ok"
        except Exception: return None,"cli: manifest unreadable"
    tail=(out.strip().splitlines() or ["no output"])[-1][:100]
    return None,"%s; cli rc=%d %s"%(kh,rc,tail)

try:
    if POLICY=="never":
        print("cache write-back disabled (H3_CACHE_UPLOAD=never)")
    else:
        # 1) inventory of local model files (symlinks = already cached from a dataset)
        inv={}
        for p in MODELS.rglob("*"):
            if p.is_file() or p.is_symlink():
                try: inv[str(p.relative_to(MODELS))]=p
                except Exception: pass
        sizes={}; total=0
        for rel,p in inv.items():
            try: sz=p.stat().st_size
            except OSError: sz=0
            sizes[rel]=sz; total+=sz
        print("cache inventory: %d files, %.1f GB"%(len(inv),total/1e9))
        if total>CAP:
            print("SKIP upload: %.1f GB >= 200 GB dataset cap - continuing"%(total/1e9))
        elif total>shutil.disk_usage("/tmp").free:
            print("SKIP upload: not enough local disk to stage - continuing")
        else:
            # 2) owner + dataset ref (kagglehub native notebook auth first)
            print("auth env: KAGGLE_USERNAME=%s KAGGLE_KEY=%s KAGGLE_API_V1_TOKEN=%s kaggle.json=%s"%(
                bool(os.environ.get("KAGGLE_USERNAME")),bool(os.environ.get("KAGGLE_KEY")),
                bool(os.environ.get("KAGGLE_API_V1_TOKEN")),
                os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json"))))
            try:
                from kagglesdk.kaggle_env import is_in_kaggle_notebook
                print("in-kaggle-notebook:",is_in_kaggle_notebook())
            except Exception as e:
                print("in-kaggle-notebook: unknown (%s)"%type(e).__name__)
            user,auth_src=_username()
            ref="%s/%s"%(user,SLUG)
            print("cache owner: %s (via %s)"%(user,auth_src))
            # 3) remote MANIFEST probe -> drift decision
            remote,probe=_remote_manifest(ref)
            print("cache dataset: %s | probe: %s"%(ref,probe))
            need=False
            if POLICY=="always":
                need=True; print("policy=always -> refresh dataset version")
            elif remote is None:
                need=True; print("remote MANIFEST unavailable -> upload (first run or probe failed)")
            else:
                local={rel:{"size":sizes[rel]} for rel in sorted(sizes) if rel.endswith(".safetensors")}
                rsize={k:{"size":(v or {}).get("size")} for k,v in sorted(remote.items())}
                if rsize!=local:
                    need=True; print("manifest drift: %d remote vs %d local files -> upload"%(len(rsize),len(local)))
                else:
                    print("cache dataset up to date (manifest match)")
            # 4) stage (hardlinks where possible - near-zero disk) then upload
            if need:
                if STAGE.exists(): shutil.rmtree(STAGE,ignore_errors=True)
                STAGE.mkdir(parents=True,exist_ok=True)
                for rel,p in inv.items():
                    dst=STAGE/rel
                    dst.parent.mkdir(parents=True,exist_ok=True)
                    if p.is_symlink():
                        try: os.symlink(os.readlink(p),dst)   # preserve original target
                        except Exception: shutil.copy2(p,dst)
                    else:
                        try: os.link(p,dst)                   # hardlink: zero extra disk
                        except Exception:
                            try: os.symlink(str(p.resolve()),dst)
                            except Exception: shutil.copy2(p,dst)
                stage_man={rel:{"size":sizes[rel]} for rel in sorted(sizes) if rel.endswith(".safetensors")}
                if (STAGE/"MANIFEST.json").exists(): (STAGE/"MANIFEST.json").unlink()
                json.dump(stage_man,open(str(STAGE/"MANIFEST.json"),"w"),indent=1)
                json.dump({"title":TITLE,"id":ref,"licenses":[{"name":"other"}]},
                          open(str(STAGE/"dataset-metadata.json"),"w"),indent=2)
                print("staged %d model files for upload -> %s"%(len(stage_man),ref))
                if POLICY=="dry":
                    print("DRY: would upload staged folder to %s"%ref)
                else:
                    up=False
                    t0=time.time()
                    try:
                        import kagglehub, inspect
                        kw={"version_notes":"auto refresh: %d files, %.1f GB"%(len(stage_man),total/1e9)}
                        if "ignore_patterns" in inspect.signature(kagglehub.dataset_upload).parameters:
                            kw["ignore_patterns"]=["dataset-metadata.json"]
                        kagglehub.dataset_upload(ref,str(STAGE),**kw)
                        print("kagglehub upload OK -> %s (%.0f s)"%(ref,time.time()-t0)); up=True
                    except Exception as e:
                        print("kagglehub upload failed (%s: %s)"%(type(e).__name__,str(e)[:200]))
                    if not up:
                        rc0,_=_kg(["datasets","metadata",ref,"-p","/tmp/h3_meta_chk"],timeout=120)
                        if rc0==0:
                            rc,out=_kg(["datasets","version","-p",str(STAGE),"-m","model cache refresh","-q"])
                            print("cli version rc=%d"%rc,out[-300:])
                        else:
                            rc,out=_kg(["datasets","create","-p",str(STAGE),"-q"])
                            print("cli create rc=%d"%rc,out[-300:])
                            if rc!=0:
                                rc,out=_kg(["datasets","version","-p",str(STAGE),"-m","model cache refresh","-q"])
                                print("cli version-after-create rc=%d"%rc,out[-300:])
                        if rc!=0: print("upload failed - run continues (never fails)")
                shutil.rmtree(STAGE,ignore_errors=True)
except Exception as e:
    print("cache write-back skipped (run continues, never fails):",e)
